🛒 Desafio 3: E-commerce Olist (Volume vs. Valor)
Célula de Texto (Markdown):

3. Análise de Vendas (E-commerce Olist)
Objetivo: Analisar o desempenho de vendas por categoria de produto, confrontando Volume de Vendas vs. Faturamento Total.

Técnicas Utilizadas:

Joins (Merges) de múltiplas tabelas relacionais (Pedidos, Itens e Produtos).

Cálculo de KPI (Ticket Médio).

Ordenação e Ranking.

Base de dados: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## 1. Carregamento dos Dados (Extract)
Importação das tabelas relacionais do E-commerce Olist.
Nesta etapa, carregamos duas tabelas principais:
* **Pedidos (`orders`):** Contém as datas e o status do pedido.
* **Itens (`order_items`):** Contém o preço e o detalhe do produto vendido.

In [ ]:
import pandas as pd

# Carregando as tabelas dimensão e fato
pedidos = pd.read_csv("olist_orders_dataset.zip")
itens_vendidos = pd.read_csv("olist_order_items_dataset.zip")

# Visualizando as primeiras linhas para entender as chaves de ligação
display(pedidos.head())
display(itens_vendidos.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


## 2. Unificação e Limpeza (Transform)
Realizamos o primeiro `merge` (Left Join) para unir os pedidos aos seus itens usando a chave `order_id`.
Em seguida, aplicamos um filtro vital de negócio: **Remover pedidos cancelados**, garantindo que a análise considere apenas vendas efetivadas.

In [7]:
# Merge 1: Unindo Pedidos com Itens
# Isso cria uma tabela onde cada linha é um item dentro de um pedido
df_completo = pd.merge(pedidos, itens_vendidos, on='order_id')

# Filtrando os Cancelados
# Regra de Negócio: Considerar apenas o que foi faturado (diferente de 'canceled')
df_validos = df_completo[df_completo['order_status'] != 'canceled']

# Validação (Sanity Check): Calculando o GMV (Gross Merchandise Value) total histórico
total_vendido = df_validos['price'].sum()
print(f"O total vendido na história foi: R$ {total_vendido:,.2f}")

O total vendido na história foi: R$ 13,496,408.43


## 3. Enriquecimento dos Dados (Produto)
A tabela de vendas possui apenas o ID do produto. Para analisar por **Categoria**, realizamos um segundo `merge` com a tabela de produtos.

In [ ]:
# Carregando a tabela de produtos para pegar os nomes das categorias
produtos = pd.read_csv("olist_products_dataset.zip")

# Merge 2: Tabela de Vendas Limpa + Tabela de Produtos
# Chave de ligação: 'product_id'
df_final = pd.merge(df_validos, produtos, on='product_id')

# Visualizando o Dataset Analítico Final
# Selecionamos apenas as colunas que importam para a análise
display(df_final[['order_id', 'price', 'product_category_name']].head())

,order_id,price,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,29.99,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,118.70,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,159.90,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,45.00,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,19.90,papelaria


## 4. Análise Estratégica: Volume vs. Receita
Aqui confrontamos duas visões:
1.  **Volume:** O que sai mais do estoque?
2.  **Receita:** O que traz mais dinheiro para o caixa?
Utilizamos `groupby` para agregar esses valores por categoria.

In [9]:
# Ranking Simples de Volume (Quantidade de linhas/itens)
display(df_final['product_category_name'].value_counts().head())

# Relatório Completo (Agregando Volume e Faturamento)
relatorio = df_final.groupby('product_category_name').agg({
    'order_id': 'count', # Contagem de vendas
    'price': 'sum'       # Soma do valor (Receita)
}).reset_index()

# Renomeando para termos de negócio
relatorio.rename(columns={
    "order_id": "Qtd_Vendas",
    "price": "Faturamento"
}, inplace=True)

# Ordenando pelo Faturamento (Top Receita)
top_faturamento = relatorio.sort_values('Faturamento', ascending=False)

# Visualizando os "Carros-Chefe" da empresa
display(top_faturamento.head())

product_category_name
cama_mesa_banho           11097
beleza_saude               9634
esporte_lazer              8590
moveis_decoracao           8298
informatica_acessorios     7781
Name: count, dtype: int64

,product_category_name,Qtd_Vendas,Faturamento
11,beleza_saude,9634,1255695.13
66,relogios_presentes,5970,1198185.21
13,cama_mesa_banho,11097,1035964.06
32,esporte_lazer,8590,979740.92
44,informatica_acessorios,7781,904322.02


## 5. Indicador de Eficiência (Ticket Médio)
Calculamos o **Ticket Médio** (Faturamento / Quantidade) para identificar categorias de alto valor agregado (Luxo/Premium).
Aplicamos um filtro de **mínimo de 100 vendas** para garantir relevância estatística e evitar distorções com produtos vendidos apenas uma vez.

In [10]:
# Cálculo do KPI
relatorio['Ticket_Medio'] = relatorio['Faturamento'] / relatorio['Qtd_Vendas']

# Análise de "Luxo": Ordenando por Ticket Médio
# Filtro: Apenas categorias com mais de 100 vendas (para evitar outliers de nicho)
luxo = relatorio[relatorio['Qtd_Vendas'] > 100].sort_values('Ticket_Medio', ascending=False)

# Resultado: Categorias que vendem menos volume, mas geram alta receita por unidade
display(luxo.head(10))

,product_category_name,Qtd_Vendas,Faturamento,Ticket_Medio
61,pcs,203,222963.13,1098.340542
29,eletrodomesticos_2,235,110649.74,470.849957
0,agro_industria_e_comercio,212,72530.47,342.124858
45,instrumentos_musicais,669,187788.44,280.700209
31,eletroportateis,671,187907.26,280.040626
71,telefonia_fixa,261,57824.21,221.548697
25,construcao_ferramentas_seguranca,189,39589.02,209.465714
66,relogios_presentes,5970,1198185.21,200.701040
19,climatizacao,295,54723.16,185.502237
56,moveis_quarto,109,20028.78,183.750275
